# Embed anything in 60 seconds

**What you'll learn.** embpy turns *any* biological entity — a gene, a drug, a
protein, a cell — into a numerical embedding through a single call,
`BioEmbedder.embed(...)`, and stores it in an AnnData you can hand straight to
the rest of your analysis.

This whole notebook runs in seconds and downloads nothing heavy: it uses a
prior-knowledge gene table and RDKit fingerprints, both computed locally.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder

embedder = BioEmbedder(device="auto", organism="human")

## One call, one gene set

Give embpy the identifiers you already have. Here we embed eight genes by
symbol using `genept` — a text-derived, prior-knowledge embedding that needs
no model download. embpy resolves each symbol and writes one vector per gene.

In [ ]:
genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT1", "IRF1", "CDK1"]

adata = ad.AnnData(
    X=np.zeros((len(genes), 1), dtype=np.float32),
    obs=pd.DataFrame({"symbol": genes}, index=genes),
)

adata = embedder.embed(
    adata,
    entity_type="gene",
    id_type="symbol",
    obs_column="symbol",
    model="genept",
    output="anndata",
    key="X_gene_genept",
)

print("embedding shape:", adata.obsm["X_gene_genept"].shape)

That's it — `adata.obsm["X_gene_genept"]` is an `(8, d)` matrix, one row per
gene, ready for clustering, similarity, or a downstream model.

## The *same* call works for every modality

Only two things change per modality: the `entity_type` and the `model`. The
rest of the call — and the AnnData you get back — is identical:

| You have | `entity_type` | A model to try |
| --- | --- | --- |
| gene symbols / Ensembl IDs | `"gene"` | `genept`, `hyenadna_small_32k` |
| protein sequences / UniProt IDs | `"protein"` | `esm2_8M`, `prot_t5_xl` |
| SMILES strings | `"molecule"` | `morgan_fp`, `chemberta2MTR` |
| free text | `"text"` | `minilm_l6_v2` |
| single cells (AnnData counts) | *(embed the cell matrix directly)* | `scvi`, `geneformer` |

For example, proteins with a small ESM-2 (a ~30 MB download the first time):

```python
prot = ad.AnnData(np.zeros((2, 1), np.float32),
                  obs=pd.DataFrame({"seq": ["MTEYKLVVVG", "ACDEFGHIKL"]},
                                   index=["MTEYKLVVVG", "ACDEFGHIKL"]))
prot = embedder.embed(prot, entity_type="protein", id_type="sequence",
                      obs_column="seq", model="esm2_8M",
                      output="anndata", key="X_esm2")
```

…or small molecules as Morgan fingerprints (no download at all):

```python
mols = ad.AnnData(np.zeros((2, 1), np.float32),
                  obs=pd.DataFrame({"smiles": ["CCO", "c1ccccc1"]}))
mols = embedder.embed(mols, entity_type="molecule", id_type="smiles",
                      obs_column="smiles", model="morgan_fp",
                      output="anndata", key="X_morgan", attach_to="obs")
```

## What you got

Every call returned the same thing: your AnnData, with an embedding matrix in
`.obsm` and its provenance recorded in `.uns`. Nothing was written to `.X`, so
your original data is untouched.

**Next:** [Where embeddings live](02_output_contract.ipynb) explains the
storage contract, and [Compare embedding spaces](03_compare_models.ipynb)
shows how to tell whether two models see your biology the same way.